In [5]:
!pip install -q transformers datasets accelerate -U
print("setup complete.")

setup complete.


Our training data

In [6]:
train_texts = [
    "This movie was absolutely wonderful, I loved every minute of it.",
    "What a fantastic performance, the whole cast was brilliant.",
    "I really enjoyed this book, could not put it down.",
    "The food at this restaurant was delicious and beautifully presented.",
    "Great service, friendly staff, and a lovely atmosphere overall.",
    "This phone exceeded my expectations in every possible way.",
    "An outstanding film with a heartwarming and memorable story.",
    "The hotel room was clean, spacious, and had a stunning view.",
    "I am thrilled with how well this laptop performs every day.",
    "The concert was incredible, the band sounded better than ever.",
    "Such a delightful little cafe, I will definitely be coming back.",
    "The customer support team resolved my issue quickly and politely.",
    "This course explained everything clearly and I learned so much.",
    "The new update makes the app so much faster and easier to use.",
    "Absolutely loved the ending, it tied the whole story together perfectly.",
    "The staff went above and beyond to make our stay comfortable.",
    "A brilliant, thought-provoking novel that I would recommend to anyone.",
    "The pizza was fresh, hot, and full of flavour.",
    "I am very happy with my purchase, it works perfectly.",
    "The trip was wonderful from start to finish.",
    "This movie was boring and the plot made no sense at all.",
    "Terrible customer service, nobody responded to my complaints.",
    "The food arrived cold and tasted quite bland.",
    "I regret buying this phone, it is slow and the battery is weak.",
    "The hotel room was dirty and smelled unpleasant.",
    "What a disappointing performance, the actors seemed uninterested.",
    "The book was confusing and I gave up halfway through.",
    "This app keeps crashing and it is extremely frustrating to use.",
    "The concert was way too loud and poorly organized.",
    "Such an unpleasant experience, the staff were rude to us.",
    "The laptop overheats constantly and the fan is very noisy.",
    "I would not recommend this restaurant, the service was awful.",
    "The ending felt rushed and left too many questions unanswered.",
    "This course was disorganized and hard to follow.",
    "The new update broke several features that used to work fine.",
    "The room was freezing and the heater did not work at all.",
    "I am very unhappy with this purchase, it broke within a week.",
    "The trip was stressful and nothing went according to plan.",
    "The pizza was soggy and clearly not freshly made.",
    "Nobody at the store seemed willing to help us at all.",
]

train_labels = [1]*20 + [0]*20

test_texts = [
    "I had a great time at the beach with my family.",
    "The new restaurant downtown serves amazing pasta.",
    "This gadget works exactly as advertised, very pleased.",
    "The lecture was engaging and easy to understand.",
    "Everything about this vacation exceeded my expectations.",
    "The product stopped working after just two days.",
    "I found the customer service painfully slow and unhelpful.",
    "The movie dragged on and felt far too long.",
    "This laptop keeps freezing and it is very annoying.",
    "The meal was overpriced and honestly quite tasteless.",
]
test_labels = [1, 1, 1, 1, 1, 0, 0, 0, 0, 0]

print(f"Training examples: {len(train_texts)}  |  Test examples: {len(test_texts)}")

Training examples: 40  |  Test examples: 10


Step -2  Tokenize the data

In [17]:
from datasets import Dataset
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_ds = Dataset.from_dict({"text": train_texts, "label": train_labels})
test_ds = Dataset.from_dict({"text": test_texts, "label": test_labels})

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=32)

train_ds = train_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)

print(train_ds[0].keys())


Map:   0%|          | 0/40 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

dict_keys(['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'])


Step 3 Load the pretrained model and add the a classification head


In [18]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
print("Model loaded. Notice the warning above about newly initialized weights -")
print("that is the brand-new classification head, which knows nothing yet!")

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded. Notice the warning above about newly initialized weights -
that is the brand-new classification head, which knows nothing yet!


Step 4 Fine-tuning

In [26]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./sentiment_model",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    logging_steps=5,
    save_strategy="no",
    report_to="none",
)

trainer = Trainer(model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
)

trainer.train()


Step,Training Loss
5,0.254350
10,0.097800
15,0.056925


TrainOutput(global_step=15, training_loss=0.13635834058125815, metrics={'train_runtime': 19.8164, 'train_samples_per_second': 6.056, 'train_steps_per_second': 0.757, 'total_flos': 993505489920.0, 'train_loss': 0.13635834058125815, 'epoch': 3.0})

Evaluate on our head-out test sentences  

In [30]:
import numpy as np

predictions = trainer.predict(test_ds)
predicted_labels = np.argmax(predictions.predictions, axis=-1)
accuracy = (predicted_labels == np.array(test_labels)).mean()

print(f"Test accuracy: {accuracy:.0%}\n")
for text, true_label, pred_label in zip(test_texts, test_labels, predicted_labels):
    mark = " " if true_label == pred_label else "X"
    print(f"{mark} true={true_label} predicted={pred_label}   {text}")

Test accuracy: 100%

  true=1 predicted=1   I had a great time at the beach with my family.
  true=1 predicted=1   The new restaurant downtown serves amazing pasta.
  true=1 predicted=1   This gadget works exactly as advertised, very pleased.
  true=1 predicted=1   The lecture was engaging and easy to understand.
  true=1 predicted=1   Everything about this vacation exceeded my expectations.
  true=0 predicted=0   The product stopped working after just two days.
  true=0 predicted=0   I found the customer service painfully slow and unhelpful.
  true=0 predicted=0   The movie dragged on and felt far too long.
  true=0 predicted=0   This laptop keeps freezing and it is very annoying.
  true=0 predicted=0   The meal was overpriced and honestly quite tasteless.
